## Imports
Importing neccesary libraries and HF API acces 

In [1]:
# ── Stdlib
import os
from operator import itemgetter

# ── LangChain core
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory, BaseChatMessageHistory
from langchain.schema.output_parser import StrOutputParser
from langchain.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage  # (only if you use explicit messages
from langchain.memory.chat_message_histories import ChatMessageHistory
from langchain.memory import ConversationBufferMemory
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


# ── LLM (Ollama)
from langchain_ollama import ChatOllama

from operator import itemgetter
from langchain_ollama import ChatOllama
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

# ── Vector store & embeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

# ── (Optional) Text splitting & reranker
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import CrossEncoder

# __ Pipeline functions 
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

import numpy as np
import pandas as pd
from datetime import datetime
import json

OMP: Warning #179: Function Can't set size of /tmp file failed:


OSError: [Errno 28] No space left on device: '/var/folders/sz/_3249w7n4qnfkxl2g0j3v2c80000gn/T/tmph71f7kty'

In [ ]:
from huggingface_hub import login
HF_API_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")
login(HF_API_TOKEN)

In [ ]:
import torch, os
torch.set_num_threads(8)  # 4, 6, 8
os.environ["OMP_NUM_THREADS"] = "8"

## Generating the answer with an LLM
We will first test the knowledge of different LLM models on hardware store products


#### Ollama local model (Free)

In [37]:
chat = ChatOllama(model="qwen2.5:1.5b", base_url="http://127.0.0.1:11434",
                  temperature=0.0, num_ctx=2048, num_predict=160, keep_alive="30m")

In [ ]:
#Test on a single question about a screwdriver
prompt = ChatPromptTemplate.from_messages([
    ("system", "Responde SIEMPRE en español neutro. No cambies de idioma."),
    ("user", "Responde en español. {question}")
])

chain = prompt | chat | StrOutputParser()

question = "¿Dame más detalles sobre el destornillador inalámbrico de la marca Truper?"
answer = chain.invoke({"question": question})
print("\n--- Assistant ---")
print(answer.strip())


--- Assistant ---
El destornillador inalámbrico de la marca Truper es un producto diseñado para facilitar la tarea del destornillado a distancia, especialmente útil en entornos donde no se puede llegar con las manos. Este tipo de herramienta utiliza una tecnología inalámbrica que permite realizar destornillados desde lejos, lo cual puede ser muy útil en situaciones como el montaje o desmontaje de dispositivos electrónicos o de computación.

El diseño del Truper es compacto y fácil de manejar, ideal para uso doméstico o profesional. La tecnología inalámbrica permite que se pueda destornillar desde una distancia considerable, lo cual puede ser especialmente útil en entornos donde no se pueden acercar


Not bad but not good

#### AutoModelForCausalLM (Local model) (Free)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

#Tried to save the model locally 
# model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name).to("cpu")

# model.save_pretrained("../models/TinyLlama")
# tokenizer.save_pretrained("../models/TinyLlama")

In [ ]:
#We charge the model
model_path = "../models/TinyLlama"

device = "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device == "mps" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa"  # usually faster; falls back if unsupported
).to(device)
model.eval()

if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token = tokenizer.eos_token


In [ ]:
#Testing on the same screwdriver question
question = "¿Dame más detalles sobre el destornillador inalámbrico de la marca Truper?"
messages = [
    {"role": "system", "content": "Responde SIEMPRE en español neutro. No cambies de idioma."},
    {"role": "user", "content": f"Responde en español. {question}"}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to(device)

with torch.inference_mode():
    out = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,              # más estable
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

new_tokens = out[0][inputs["input_ids"].shape[1]:]
resp = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
print(resp)

Sí, sí.

El destornillador inalámbrico de la marca Truper es un dispositivo de seguridad que puede ayudar a proteger tu casa de robos y daños. Este dispositivo funciona inalámbricamente y no requiere ningún cable o conexión.

El destornillador inalámbrico de Truper es un dispositivo de seguridad que puede ayudar a proteger tu casa de robos y daños. El dispositivo funciona inalámbricamente y no requiere ningún cable o conexión.

El destornillador inalámbrico de Truper


Too heavy, and inference takes too long, and overall not good 

#### HuggingFaceEndpoint (Spends Inference tokens)

In [ ]:
#Using the HuggingFaceEndpoint
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-70B-Instruct",
    task="conversational", #THIS MODEL ONLY WORKS WITH CONVERSATIONAL TASK
    temperature=0.7,
    max_new_tokens=512,
)

# ChatHuggingFace
chat = ChatHuggingFace(llm=llm)

prompt = ChatPromptTemplate.from_template("Responde en español: {question}")
parser = StrOutputParser()

chain = prompt | chat | parser

In [12]:
response = chain.invoke({"question": "¿Dame mas detalles sobre el destornillador inalambrico de la marca Truper?"})
print(response)

¡Claro! El destornillador inalámbrico de la marca Truper es una herramienta versátil y potente que ofrece una gran comodidad y flexibilidad al usuario. A continuación, te proporciono algunos detalles adicionales sobre este producto:

**Características**

* Potente motor: El destornillador inalámbrico de Truper cuenta con un motor de alta potencia que entrega hasta 300 watts de energía, lo que le permite manejar tareas pesadas con facilidad.
* Batería recargable: La batería es recargable y ofrece una autonomía prolongada, lo que te permite trabajar durante horas sin necesidad de recargar.
* Varias velocidades: El destornillador tiene varias velocidades para adaptarse a diferentes tipos de tareas, desde trabajos delicados hasta tareas más pesadas.
* Mango ergonómico: El mango es ergonómico y cómodo, lo que reduce la fatiga y te permite trabajar durante períodos prolongados.
* Accesorios incluidos: El destornillador viene con una variedad de accesorios, como bits y brocas, lo que te permi

It kind of tries to make something up (is it correct, that we will see)

## Defining our RAG

### DATA

In [41]:
# WE GET THE DESCRIPTIONS FOR ALL PRODUCTS
import ast
electric_tool_truper = pd.read_csv("../data/scrapping/sample_electric_truper_products.csv", sep=";")
# electric_tool_truper_des = electric_tool_truper["Descripcion"].apply(lambda x: ast.literal_eval(x)["descripcion"]).tolist() #avant 
electric_tool_truper_des = electric_tool_truper["Descripcion"].tolist()
df_desc = pd.DataFrame(electric_tool_truper_des, columns=['Text'])
df_desc

,Text
0,"Marca Truper, Línea 18153, Modelo 18153, Tipo ..."
1,"Marca Truper, Modelo LLM-6L, Tipo de producto ..."
2,"Marca Truper, Modelo Torx-7l, Tipo de llave Co..."
3,"Marca Truper, Modelo MAND-7/16, Tipo de mandri..."
4,"Marca Truper, Modelo PICA-X, Tipo de producto ..."
...,...
89,"Marca Truper, Modelo 14182 ""Grata De Copa 3""""..."
90,"Marca Truper, Modelo ALLX-7M, Tipo de llave Al..."
91,"""Marca Truper, Modelo PPC-11R, Tipo de punta C..."
92,"""Marca Truper, Modelo JOY-6, Formato de venta ..."


### Generating the embeddings
Embeddings are numerical representations of information. There are several embedding providers, but we'll use Hugging Face, and for the vector database, we'll use Chroma for now.

In [62]:
"""TO SAVE THE DB"""

# The embedding function
embedding_function = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# We create the Chroma vectorstore 
DB_DIR = "./chroma_db"
COLLECTION = "electric_tools_sample"

vectorstore = Chroma(
    persist_directory=DB_DIR,
    collection_name=COLLECTION,
    embedding_function=embedding_function, 
)

# # We then add texts to the index and persist
# vectorstore.add_texts(texts=df_desc["Text"].tolist()) # metadatas
# vectorstore.persist() #write the in-memory database to disk so it survives after your Python script ends

In [ ]:
#To verify db creation
print("Number of documents in the database:", len(vectorstore.get()['documents']))

docs_info = vectorstore.get()
print("IDs :", docs_info['ids'][:5])  # displays the first 5 IDs
print("Contents :", docs_info['documents'][:1])  # displays the first document
print("Metadata :", docs_info['metadatas'][:1])

Number of documents in the database: 94
IDs : ['f33926bf-c2d1-431d-a1ba-a03cc3acabec', '97b00f6d-a36f-4b92-8bb8-755636596b97', '31a6c674-1820-4230-917c-96948b36eb92', '8703eb93-8ecc-4b27-9a95-2ba4659ad821', '1443594f-a597-4ab5-bfca-f049993b72a2']
Contents : ['Marca Truper, Línea 18153, Modelo 18153, Tipo de producto Destornillador, Tipo de destornillador eléctrico Compacto, Es inalámbrico Sí, Tamaño del mandril 10 mm, Encastre 3/8, Torque máximo 22 Nm, Velocidad mínima de rotación 350 rpm, Velocidad máxima de rotación 1.200 rpm, Accesorios incluidos Batería  Diseño ligero para mayor comodidadDoble engranaje con selector de 2 velocidades mecánicas, botón de dirección de giro y bloqueo del interruptorBroquero de cambio rápido con seguro de retenciónLuz LED para iluminar área de trabajo Indicador de nivel de carga de batería.']
Metadata : [None]


In [63]:
"""TO QUERY THE DB"""
# Retriever to query the vectorstore
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

query = "Dame información sobre un destornillador inalámbrico Truper"
docs = retriever.get_relevant_documents(query)

print(f"🔎 Retrieved {len(docs)} docs")
for i, d in enumerate(docs, 1):
    print(f"\n— Doc {i} —")
    print("Metadata:", d.metadata)
    print(d.page_content[:500], "...")

🔎 Retrieved 5 docs

— Doc 1 —
Metadata: {}
Marca Truper, Línea 18153, Modelo 18153, Tipo de producto Destornillador, Tipo de destornillador eléctrico Compacto, Es inalámbrico Sí, Tamaño del mandril 10 mm, Encastre 3/8, Torque máximo 22 Nm, Velocidad mínima de rotación 350 rpm, Velocidad máxima de rotación 1.200 rpm, Accesorios incluidos Batería  Diseño ligero para mayor comodidadDoble engranaje con selector de 2 velocidades mecánicas, botón de dirección de giro y bloqueo del interruptorBroquero de cambio rápido con seguro de retenciónLuz ...

— Doc 2 —
Metadata: {}
Marca Truper, Modelo 12638, Formato de venta Unidad, Largo de la cinta 20 m, Ancho de la cinta 15 mm, Peso 310 g, Unidades de medida cm, ft, Material de la cinta Fibra de vidrio, Material de la superficie Plástico, Es retráctil Sí, Es plegable Sí, Con botón de tranca Sí, Con gancho magnético Sí, Con diseño ergonómico Sí  Cinta larga cubierta con carcasa fabricada en ABS con TPR resistente a impactos.  Cinta de fibra de vidri

It is actually able to get the product's informations

#### Looking for the most relevant chunk

In [45]:
query = "destornillador eléctrico inalámbrico de la marca Truper"

retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
best_passage = retriever.get_relevant_documents(query)

if best_passage:
    print(best_passage[0].page_content)
else:
    print("Aucun document trouvé pour la requête.")


Marca Truper, Modelo 12638, Formato de venta Unidad, Largo de la cinta 20 m, Ancho de la cinta 15 mm, Peso 310 g, Unidades de medida cm, ft, Material de la cinta Fibra de vidrio, Material de la superficie Plástico, Es retráctil Sí, Es plegable Sí, Con botón de tranca Sí, Con gancho magnético Sí, Con diseño ergonómico Sí  Cinta larga cubierta con carcasa fabricada en ABS con TPR resistente a impactos.  Cinta de fibra de vidrio recubierta de PVC que permite limpiarse fácilmente.  Graduada por ambos lados.  Especificaciones:  Ref: TP20ME  Longitud: 20 metros - 66 ft  Ancho de la cinta: 14.5 mm  Espesor de la cinta: 0.47 mm  Escala: cm - m / in - ft  Color de la cinta: Amarillo  Peso: 310 gramos  Cinta de fibra de vidrio recubierta de PVC que permite que pueda mojarse y limpiarse fácilmente.  Carcasa de ABS con TPR resistente a impactos.  Graduada por ambos lados.  Ideal para: Construcción, ingeniería civil, etc. Lo que tienes que saber de este producto, Unidades por pack: 1Formato de vent

## Generating the answer using the LLMs

### Using HF endpoint

In [28]:
#Modelo Fallback
llm_fallback = HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    task="conversational", #THIS MODEL ONLY WORKS WITH CONVERSATIONAL TASK
    temperature=0.7,
    max_new_tokens=512,
)

chat_fallback = ChatHuggingFace(llm=llm_fallback)

In [ ]:
#For the memory
memory = ConversationBufferMemory(return_messages=True)

# Prompt con contexto incluido en el mensaje del usuario
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente experto en herramientas eléctricas de ferretería. Usa el contexto proporcionado. Si no sabes, di: 'No tengo suficiente información'."),
    ("user", "Contexto: {context}\n\nPregunta: {question}")
])

# Primary y fallback
primary_chain = prompt | chat | StrOutputParser()
fallback_chain = prompt | chat_fallback | StrOutputParser()
main_chain = primary_chain.with_fallbacks([fallback_chain])

# Parallel input: context + query
#Parce que dans LangChain, lorsque tu passes un Retriever (comme retriever) dans une RunnableParallel ou RunnableMap, il est automatiquement appelé avec .get_relevant_documents(query).
rag_chain = RunnableParallel({
    "context": itemgetter("question") | retriever,
    "question": RunnablePassthrough()
}) | main_chain


# For history we use InMemoryChatMessageHistory
chat_histories = {}

def get_chat_history(session_id: str = "default") -> ChatMessageHistory:
    if session_id not in chat_histories:
        # Aquí usamos InMemory pero bajo la interfaz ChatMessageHistory
        chat_histories[session_id] = InMemoryChatMessageHistory()
    return chat_histories[session_id]

# Chain and history of the chat 
rag_with_memory = RunnableWithMessageHistory(
    rag_chain,
    get_chat_history,
    input_messages_key="question",
    history_messages_key="history"
)

def answer(query, session_id="default"):
    return rag_with_memory.invoke(
        {"question": query},
        config={"configurable": {"session_id": session_id}}
    )
#Example
consulta = "Dame información sobre algún destornillador eléctrico inalámbrico de la marca Truper"
respuesta = answer(consulta)
print("🧠 Pregunta:", consulta)
print("💬 Respuesta:", respuesta)

🧠 Pregunta: Dame información sobre algún destornillador eléctrico inalámbrico de la marca Truper
💬 Respuesta: Excelente elección! Tengo justo la información que necesitas. Según mi base de datos, el destornillador eléctrico inalámbrico de la marca Truper que te puedo recomendar es el modelo 18153, que forma parte de la línea 18153.

Este destornillador eléctrico compacto cuenta con varias características destacadas:

* Es inalámbrico, lo que te brinda libertad de movimiento y facilidad de uso.
* Tiene un tamaño de mandril de 10 mm y un encastre de 3/8.
* Ofrece un torque máximo de 22 Nm y una velocidad de rotación variable entre 350 rpm y 1.200 rpm.
* Incluye una batería y tiene un diseño ligero para mayor comodidad.
* Cuenta con doble engranaje con selector de 2 velocidades mecánicas, botón de dirección de giro y bloqueo del interruptor.
* Tiene un broquero de cambio rápido con seguro de retención y una luz LED para iluminar el área de trabajo.
* Además, cuenta con un indicador de niv

Quite good!! But sadly we will be using HF inference credit

### RAG Using Transformers pipeline

Using pipeline the model is downloaded

In [ ]:
#LLM pipeline 
llm_pipeline = pipeline(
    model="meta-llama/Meta-Llama-3-8B-Instruct",  # ou plus petit
    task="text-generation",
    model_kwargs={"temperature": 0.7, "max_new_tokens": 512},
    device_map="auto"
)
chat = HuggingFacePipeline(pipeline=llm_pipeline)

fallback_pipeline = pipeline(
    model="meta-llama/Meta-Llama-3-8B-Instruct",  # même ou autre modèle local
    task="text-generation",
    model_kwargs={"temperature": 0.7, "max_new_tokens": 512},
    device_map="auto"
)
chat_fallback = HuggingFacePipeline(pipeline=fallback_pipeline)


Using pipeline is free but downloading such a big model is no efficient and when used for inference with a CPU it is really slow

### Ollama 

In [77]:
embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

vectorstore = Chroma(
    persist_directory="./chroma_db",
    collection_name="electric_tools_sample",
    embedding_function=embedding_function
)
retriever = vectorstore.as_retriever() 

In [78]:

#Ollama models (primary + fallback) 
OLLAMA_BASE = os.environ.get("OLLAMA_BASE_URL", "http://127.0.0.1:11434")

chat = ChatOllama(
    model="qwen2.5:1.5b",  #qwen2.5:3b              # good Spanish + quality
    base_url=OLLAMA_BASE,
    temperature=0.0,                  # deterministic & faster for RAG
    num_ctx=2048,
    num_predict=160,                  # cap output length to reduce latency
    keep_alive="30m",
    timeout=120,
)

chat_fallback = ChatOllama(
    model="qwen2.5:1.5b",             # faster fallback
    base_url=OLLAMA_BASE,
    temperature=0.0,
    num_ctx=2048,
    num_predict=140,
    keep_alive="30m",
    timeout=120,
)

In [79]:
# ---- quick retrieval sanity check (run once) ----
_test_q = "Dame información sobre un destornillador inalámbrico Truper"
_docs = retriever.get_relevant_documents(_test_q)
print(f"[debug] retrieved {_docs and len(_docs) or 0} docs")
if _docs:
    print("[debug] first doc snippet:", _docs[0].page_content[:120])

[debug] retrieved 4 docs
[debug] first doc snippet: Marca Truper, Línea 18153, Modelo 18153, Tipo de producto Destornillador, Tipo de destornillador eléctrico Compacto, Es 


In [80]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente experto en herramientas eléctricas de ferretería. "
               "Usa el contexto proporcionado de manera natural y amable. "
               "Si no sabes, di: 'No tengo suficiente información'. "
               "Responde SIEMPRE en español neutro."),
    ("user", "Contexto:\n{context}\n\nPregunta: {question}")
])

primary_chain = prompt | chat | StrOutputParser()
fallback_chain = prompt | chat_fallback | StrOutputParser()
main_chain = primary_chain.with_fallbacks([fallback_chain])

def format_docs(docs):
    # docs is a List[Document]; join only the text
    return "\n\n".join(f"- {d.page_content}" for d in docs)

"""	itemgetter("question") | retriever
  •	takes the "question" string from the input dict,
  •	passes it to the retriever,
  •	the retriever runs get_relevant_documents(question) internally,
  •	and returns List[Document]."""

rag_chain = (
    RunnableParallel({
        "context": itemgetter("question") | retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    })
    | main_chain
)

# memory (same style as your working version)
chat_histories = {}

def get_chat_history(session_id: str = "default") -> BaseChatMessageHistory:
    if session_id not in chat_histories:
        chat_histories[session_id] = InMemoryChatMessageHistory()
    return chat_histories[session_id]

rag_with_memory = RunnableWithMessageHistory(
    rag_chain,
    get_chat_history,
    input_messages_key="question",
    history_messages_key="history"
)

def log_conversation(session_id, question, answer):
    os.makedirs("logs", exist_ok=True)
    log_entry = {
        "session_id": session_id,
        "question": question,
        "answer": answer,
        "timestamp": datetime.now().isoformat()
    }
    with open("logs/conversations.jsonl", "a") as f:
        f.write(json.dumps(log_entry, ensure_ascii=False) + "\n")

def answer(query, session_id="default"):
    response = rag_with_memory.invoke(
        {"question": query},
        config={"configurable": {"session_id": session_id}}
    )
    log_conversation(session_id, query, response)
    return response

In [ ]:
q = "Dame información sobre algún destornillador eléctrico inalámbrico de la marca Truper"
res = answer(q)
print(res)

El destornillador eléctrico compacto de la marca Truper, modelo 18153, es un producto diseñado para proporcionar una comodidad y eficiencia en el trabajo. Con un tamaño del mandril de 10 mm, encastre de 3/8, torque máximo de 22 Nm y velocidad máxima de rotación de 1.200 rpm, este destornillador es ideal para realizar trabajos precisos con una facilidad considerable.

Además, cuenta con un diseño ligero que facilita la movilidad durante el trabajo, lo cual puede ser especialmente útil en entornos de construcción o reparación donde se requiere rapidez y eficiencia. El destornillador es in


`We found our architecture for the V0`

PostgreSQL